# Lab 3 — Tester et organiser un tournoi (correction)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-medecine/notebooks/pym-s8-lab-3-correction.ipynb)

*Python pour la médecine — Séance 8, projet Puissance 4.*


**Objectifs.** Concevoir une stratégie de test ; écrire des tests qui
**détectent** de vraies erreurs ; tester l'agent sur des scénarios ;
écrire un module et ses tests `pytest`, mesurer sa qualité avec
`pylint` ; mesurer le temps par coup et la mémoire ; comparer les
agents dans un tournoi et savoir combien de parties il faut pour
conclure.

**Comment travailler.** Exécutez les cellules dans l'ordre. Une cellule
marquée `# À vous` contient un squelette (`...`) à compléter ; la
cellule suivante vérifie votre réponse et affiche « ✅ Exercice N
réussi ». Les cellules « Lire du code » demandent d'écrire votre
prédiction **avant** d'exécuter.

## 0. Installation

In [ ]:
import subprocess
import sys

try:
    import pettingzoo
except ImportError:  # Colab : PettingZoo n'est pas préinstallé
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pettingzoo[classic]"], check=True)

import random
import time

import matplotlib.pyplot as plt
import numpy as np
from pettingzoo.classic import connect_four_v3

random.seed(0)
print("PettingZoo prêt")

### Le code des labs précédents

La cellule suivante reprend la correction des labs 1 et 2 : exécutez-la
sans la modifier (ou remplacez-la par votre propre code).

In [ ]:
def afficher_plateau(observation):
    """Le plateau en texte : X = mes pions, O = l'adversaire, . = vide."""
    obs = np.asarray(observation)
    lignes = []
    for ligne in range(6):
        cases = []
        for col in range(7):
            if obs[ligne, col, 0] == 1:
                cases.append("X")
            elif obs[ligne, col, 1] == 1:
                cases.append("O")
            else:
                cases.append(".")
        lignes.append(" ".join(cases))
    return "\n".join(lignes)

def colonnes_libres(action_mask):
    """Les indices des colonnes autorisées (masque à 1)."""
    return [col for col, autorise in enumerate(action_mask) if autorise]

class AgentAleatoire:
    """Joue une colonne autorisée au hasard (le contrat ML-Arena)."""

    def setup(self, observation_space, action_space):
        return True

    def reset(self, env_player_name, episode_index):
        self.env_player_name = env_player_name
        self.episode_index = episode_index
        return True

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        return random.choice(colonnes_libres(action_mask))

class AgentPondere(AgentAleatoire):
    """Au hasard, mais les colonnes du centre sont plus probables."""

    POIDS = [1, 2, 3, 4, 3, 2, 1]

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        colonnes = colonnes_libres(action_mask)
        poids = [self.POIDS[col] for col in colonnes]
        return random.choices(colonnes, weights=poids)[0]

def jouer_partie(agent_a, agent_b, a_commence=True, index=0):
    """Une partie appelée comme sur ML-Arena : (résultat de agent_a, nb de coups).

    Le résultat vaut +1 (agent_a gagne), 0 (nulle) ou -1 (agent_a perd).
    """
    env = connect_four_v3.env()
    env.reset()
    if a_commence:
        places = {"player_0": agent_a, "player_1": agent_b}
    else:
        places = {"player_0": agent_b, "player_1": agent_a}
    for nom, agent in places.items():
        agent.reset(nom, index)
    gains = {nom: 0.0 for nom in env.possible_agents}
    nb_coups = 0
    for nom in env.agent_iter():
        observation, reward, terminated, truncated, info = env.last()
        if terminated or truncated:
            env.step(None)  # un joueur dont la partie est finie sort ainsi
            continue
        action = places[nom].choose_action(
            observation["observation"], float(reward), False, False, info,
            action_mask=observation["action_mask"])
        env.step(action)
        nb_coups += 1
        for joueur, gain in env.rewards.items():
            gains[joueur] += gain
    env.close()
    nom_a = "player_0" if a_commence else "player_1"
    return gains[nom_a], nb_coups

def jouer_match(agent_a, agent_b, n_parties=100):
    """n parties, agent_a commence une partie sur deux. Bilan vu par agent_a."""
    env = connect_four_v3.env()
    for agent in (agent_a, agent_b):  # une fois par match, comme sur ML-Arena
        agent.setup(env.observation_space("player_0"),
                    env.action_space("player_0"))
    bilan = {"victoires": 0, "nuls": 0, "defaites": 0, "coups": []}
    for i in range(n_parties):
        resultat, nb_coups = jouer_partie(agent_a, agent_b,
                                          a_commence=(i % 2 == 0), index=i)
        if resultat > 0:
            bilan["victoires"] += 1
        elif resultat < 0:
            bilan["defaites"] += 1
        else:
            bilan["nuls"] += 1
        bilan["coups"].append(nb_coups)
    return bilan


LIGNES, COLONNES = 6, 7
VIDE, MOI, ADV = 0, 1, 2

def vers_grille(observation):
    """(6, 7, 2) -> grille 6 x 7 : 0 = vide, 1 = moi, 2 = l'adversaire."""
    obs = np.asarray(observation)
    return (obs[:, :, 0] + 2 * obs[:, :, 1]).astype(int)

def grille_depuis_texte(texte):
    """Une grille depuis 6 lignes de texte : X = moi, O = l'adversaire."""
    symboles = {".": VIDE, "X": MOI, "O": ADV}
    lignes = [l.split() for l in texte.strip().splitlines()]
    return np.array([[symboles[s] for s in l] for l in lignes], dtype=int)


def observation_depuis_grille(grille):
    """L'inverse de vers_grille : l'observation (6, 7, 2) que reçoit l'agent."""
    return np.stack([grille == MOI, grille == ADV], axis=-1).astype(np.int8)


def masque_depuis_grille(grille):
    """Le masque des colonnes non pleines, comme action_mask."""
    return (grille[0] == VIDE).astype(np.int8)

def coups_valides(grille):
    """Les colonnes dont la case du haut est vide."""
    return [col for col in range(COLONNES) if grille[0, col] == VIDE]

def ligne_libre(grille, col):
    """La ligne où tombe un pion lâché dans col, ou None si elle est pleine."""
    for ligne in range(LIGNES - 1, -1, -1):  # du bas (5) vers le haut (0)
        if grille[ligne, col] == VIDE:
            return ligne
    return None

def simuler_coup(grille, col, joueur):
    """Une COPIE de la grille où joueur a joué dans col."""
    nouvelle = grille.copy()
    nouvelle[ligne_libre(grille, col), col] = joueur
    return nouvelle

def construire_fenetres():
    """Toutes les suites de 4 cases alignées sur la grille 6 x 7."""
    fenetres = []
    for ligne in range(LIGNES):
        for col in range(COLONNES):
            for dl, dc in [(0, 1), (1, 0), (1, 1), (-1, 1)]:
                cases = [(ligne + k * dl, col + k * dc) for k in range(4)]
                if all(0 <= l < LIGNES and 0 <= c < COLONNES for l, c in cases):
                    fenetres.append(cases)
    return fenetres


FENETRES = construire_fenetres()

def est_gagnant(grille, joueur):
    """Vrai si joueur a quatre pions alignés quelque part."""
    for fenetre in FENETRES:
        if all(grille[l, c] == joueur for l, c in fenetre):
            return True
    return False

def coup_gagnant(grille, joueur):
    """Une colonne qui fait gagner joueur tout de suite, sinon None."""
    for col in coups_valides(grille):
        if est_gagnant(simuler_coup(grille, col, joueur), joueur):
            return col
    return None

def coups_surs(grille, coups):
    """Les coups après lesquels l'adversaire ne peut pas gagner tout de suite."""
    return [col for col in coups
            if coup_gagnant(simuler_coup(grille, col, MOI), ADV) is None]

class AgentRegles(AgentAleatoire):
    """Gagner, sinon bloquer, sinon un coup sûr de préférence au centre."""

    POIDS = [1, 2, 3, 4, 3, 2, 1]

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        grille = vers_grille(observation)
        coups = colonnes_libres(action_mask)
        for joueur in (MOI, ADV):  # 1. gagner, 2. bloquer
            col = coup_gagnant(grille, joueur)
            if col is not None:
                return col
        candidats = coups_surs(grille, coups) or coups  # 3. ne pas offrir
        poids = [self.POIDS[col] for col in candidats]  # 4. le centre
        return random.choices(candidats, weights=poids)[0]

---

## 1. Un plan de test

| catégorie | question | comment | critère de succès |
|---|---|---|---|
| fonctionnel | les fonctions font-elles ce qu'elles disent ? | grilles choisies, `assert` | 100 % des tests passent |
| fonctionnel | l'agent joue-t-il toujours un coup autorisé ? | positions aléatoires | aucun coup interdit |
| stratégique | gagne-t-il, bloque-t-il quand il faut ? | scénarios | le coup attendu |
| stratégique | est-il plus fort qu'un autre agent ? | match de N parties | ≥ 90 % contre l'aléatoire |
| performance | répond-il à temps ? | chronomètre | < 0,1 s par coup (limite : 0.25 s) |
| performance | tient-il en mémoire ? | `tracemalloc` | quelques Mo |

Un test utile est un test qui **échoue** quand le code est faux. On le
vérifie en lui donnant du code volontairement faux (un *mutant*).

### Exercice 1 — Des tests qui attrapent les erreurs

Écrivez trois fonctions de test, chacune avec plusieurs `assert` :
`test_coups_valides`, `test_ligne_libre` et `test_est_gagnant`. La
cellule de vérification les exécute sur le bon code (ils doivent
passer) puis sur trois mutants (ils doivent échouer) : un
`coups_valides` qui ignore les colonnes pleines, un `ligne_libre` qui
remplit par le haut, un `est_gagnant` qui oublie les diagonales.

In [ ]:
def test_coups_valides():
    vide = np.zeros((6, 7), dtype=int)
    assert coups_valides(vide) == list(range(7))
    pleine = vide.copy()
    pleine[:, 2] = [MOI, ADV] * 3
    assert coups_valides(pleine) == [0, 1, 3, 4, 5, 6]


def test_ligne_libre():
    vide = np.zeros((6, 7), dtype=int)
    assert ligne_libre(vide, 0) == 5
    assert ligne_libre(simuler_coup(vide, 0, MOI), 0) == 4
    pleine = vide.copy()
    pleine[:, 2] = [MOI, ADV] * 3
    assert ligne_libre(pleine, 2) is None


def test_est_gagnant():
    g = np.zeros((6, 7), dtype=int)
    assert not est_gagnant(g, MOI)
    g[5, 0:4] = MOI
    assert est_gagnant(g, MOI) and not est_gagnant(g, ADV)
    diag = np.zeros((6, 7), dtype=int)
    for k in range(4):
        diag[5 - k, k] = ADV          # diagonale ↗
    assert est_gagnant(diag, ADV)
    anti = np.zeros((6, 7), dtype=int)
    for k in range(4):
        anti[2 + k, k] = MOI          # diagonale ↘
    assert est_gagnant(anti, MOI)

In [ ]:
def _attrape(test, nom, mutant):
    original = globals()[nom]
    globals()[nom] = mutant
    try:
        test()
        return False
    except Exception:
        return True
    finally:
        globals()[nom] = original

def _mutant_coups(grille):
    return list(range(COLONNES))

def _mutant_ligne(grille, col):
    for ligne in range(LIGNES):
        if grille[ligne, col] == VIDE:
            return ligne
    return None

def _mutant_gagnant(grille, joueur):
    droites = [f for f in FENETRES if len({l for l, _ in f}) == 1
               or len({c for _, c in f}) == 1]
    return any(all(grille[l, c] == joueur for l, c in f) for f in droites)

for test in (test_coups_valides, test_ligne_libre, test_est_gagnant):
    test()  # sur le bon code : ne doit rien lever
assert _attrape(test_coups_valides, "coups_valides", _mutant_coups), \
    "test_coups_valides ne voit pas une colonne pleine jouée"
assert _attrape(test_ligne_libre, "ligne_libre", _mutant_ligne), \
    "test_ligne_libre ne voit pas un pion posé en haut"
assert _attrape(test_est_gagnant, "est_gagnant", _mutant_gagnant), \
    "test_est_gagnant ne teste pas les diagonales"
print("✅ Exercice 1 réussi")

### Exercice 2 — Des scénarios pour l'agent

Un scénario est une position dessinée en texte et l'ensemble des coups
acceptables. `verifier_scenario` (donnée) demande 20 fois son coup à
l'agent — il tire parfois au hasard — et vérifie qu'il est toujours
acceptable. Deux scénarios sont fournis ; ajoutez-en **au moins
trois** : une victoire verticale, un blocage en diagonale, et un piège
à éviter (l'ensemble attendu contient alors tous les coups sauf le
mauvais).

In [ ]:
def verifier_scenario(agent, texte, attendus, essais=20):
    grille = grille_depuis_texte(texte)
    for _ in range(essais):
        coup = agent.choose_action(observation_depuis_grille(grille),
                                   action_mask=masque_depuis_grille(grille))
        if coup not in attendus:
            return False
    return True

In [ ]:
SCENARIOS = [
    ("gagner à l'horizontale", """
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
X X X . O O O
""", {3}),
    ("bloquer à l'horizontale", """
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . X .
O O O . X X .
""", {3}),
    ("gagner à la verticale", """
. . . . . . .
. . . . . . .
. . . . . . .
. X . . . . .
. X . O . . .
. X . O O . .
""", {1}),
    ("bloquer en diagonale", """
. . . . . . .
. . . . . . .
. . . . . . .
. . O X . . .
. O X X . . .
O X X O . . .
""", {3}),
    ("ne pas offrir la case (4, 0)", """
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. O O O X . .
. X O X X O .
""", {1, 2, 3, 4, 5, 6}),
]

In [ ]:
assert len(SCENARIOS) >= 5, "au moins trois scénarios de plus"
agent = AgentRegles()
for nom, texte, attendus in SCENARIOS:
    assert verifier_scenario(agent, texte, attendus), f"échec : {nom}"
    print("ok :", nom)
print("✅ Exercice 2 réussi")

---

## 2. Un module, ses tests, pytest et pylint

Dans le projet, le code vit dans des fichiers `.py`, pas dans un
notebook. La cellule suivante écrit le module `puissance4.py` (le code
des labs 1 et 2) avec `%%writefile`.

In [ ]:
%%writefile puissance4.py
"""Puissance 4 : outils de grille et agents (labs 1 et 2)."""
import random

import numpy as np
from pettingzoo.classic import connect_four_v3


def afficher_plateau(observation):
    """Le plateau en texte : X = mes pions, O = l'adversaire, . = vide."""
    obs = np.asarray(observation)
    lignes = []
    for ligne in range(6):
        cases = []
        for col in range(7):
            if obs[ligne, col, 0] == 1:
                cases.append("X")
            elif obs[ligne, col, 1] == 1:
                cases.append("O")
            else:
                cases.append(".")
        lignes.append(" ".join(cases))
    return "\n".join(lignes)

def colonnes_libres(action_mask):
    """Les indices des colonnes autorisées (masque à 1)."""
    return [col for col, autorise in enumerate(action_mask) if autorise]

class AgentAleatoire:
    """Joue une colonne autorisée au hasard (le contrat ML-Arena)."""

    def setup(self, observation_space, action_space):
        return True

    def reset(self, env_player_name, episode_index):
        self.env_player_name = env_player_name
        self.episode_index = episode_index
        return True

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        return random.choice(colonnes_libres(action_mask))

class AgentPondere(AgentAleatoire):
    """Au hasard, mais les colonnes du centre sont plus probables."""

    POIDS = [1, 2, 3, 4, 3, 2, 1]

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        colonnes = colonnes_libres(action_mask)
        poids = [self.POIDS[col] for col in colonnes]
        return random.choices(colonnes, weights=poids)[0]

def jouer_partie(agent_a, agent_b, a_commence=True, index=0):
    """Une partie appelée comme sur ML-Arena : (résultat de agent_a, nb de coups).

    Le résultat vaut +1 (agent_a gagne), 0 (nulle) ou -1 (agent_a perd).
    """
    env = connect_four_v3.env()
    env.reset()
    if a_commence:
        places = {"player_0": agent_a, "player_1": agent_b}
    else:
        places = {"player_0": agent_b, "player_1": agent_a}
    for nom, agent in places.items():
        agent.reset(nom, index)
    gains = {nom: 0.0 for nom in env.possible_agents}
    nb_coups = 0
    for nom in env.agent_iter():
        observation, reward, terminated, truncated, info = env.last()
        if terminated or truncated:
            env.step(None)  # un joueur dont la partie est finie sort ainsi
            continue
        action = places[nom].choose_action(
            observation["observation"], float(reward), False, False, info,
            action_mask=observation["action_mask"])
        env.step(action)
        nb_coups += 1
        for joueur, gain in env.rewards.items():
            gains[joueur] += gain
    env.close()
    nom_a = "player_0" if a_commence else "player_1"
    return gains[nom_a], nb_coups

def jouer_match(agent_a, agent_b, n_parties=100):
    """n parties, agent_a commence une partie sur deux. Bilan vu par agent_a."""
    env = connect_four_v3.env()
    for agent in (agent_a, agent_b):  # une fois par match, comme sur ML-Arena
        agent.setup(env.observation_space("player_0"),
                    env.action_space("player_0"))
    bilan = {"victoires": 0, "nuls": 0, "defaites": 0, "coups": []}
    for i in range(n_parties):
        resultat, nb_coups = jouer_partie(agent_a, agent_b,
                                          a_commence=(i % 2 == 0), index=i)
        if resultat > 0:
            bilan["victoires"] += 1
        elif resultat < 0:
            bilan["defaites"] += 1
        else:
            bilan["nuls"] += 1
        bilan["coups"].append(nb_coups)
    return bilan


LIGNES, COLONNES = 6, 7
VIDE, MOI, ADV = 0, 1, 2

def vers_grille(observation):
    """(6, 7, 2) -> grille 6 x 7 : 0 = vide, 1 = moi, 2 = l'adversaire."""
    obs = np.asarray(observation)
    return (obs[:, :, 0] + 2 * obs[:, :, 1]).astype(int)

def grille_depuis_texte(texte):
    """Une grille depuis 6 lignes de texte : X = moi, O = l'adversaire."""
    symboles = {".": VIDE, "X": MOI, "O": ADV}
    lignes = [l.split() for l in texte.strip().splitlines()]
    return np.array([[symboles[s] for s in l] for l in lignes], dtype=int)


def observation_depuis_grille(grille):
    """L'inverse de vers_grille : l'observation (6, 7, 2) que reçoit l'agent."""
    return np.stack([grille == MOI, grille == ADV], axis=-1).astype(np.int8)


def masque_depuis_grille(grille):
    """Le masque des colonnes non pleines, comme action_mask."""
    return (grille[0] == VIDE).astype(np.int8)

def coups_valides(grille):
    """Les colonnes dont la case du haut est vide."""
    return [col for col in range(COLONNES) if grille[0, col] == VIDE]

def ligne_libre(grille, col):
    """La ligne où tombe un pion lâché dans col, ou None si elle est pleine."""
    for ligne in range(LIGNES - 1, -1, -1):  # du bas (5) vers le haut (0)
        if grille[ligne, col] == VIDE:
            return ligne
    return None

def simuler_coup(grille, col, joueur):
    """Une COPIE de la grille où joueur a joué dans col."""
    nouvelle = grille.copy()
    nouvelle[ligne_libre(grille, col), col] = joueur
    return nouvelle

def construire_fenetres():
    """Toutes les suites de 4 cases alignées sur la grille 6 x 7."""
    fenetres = []
    for ligne in range(LIGNES):
        for col in range(COLONNES):
            for dl, dc in [(0, 1), (1, 0), (1, 1), (-1, 1)]:
                cases = [(ligne + k * dl, col + k * dc) for k in range(4)]
                if all(0 <= l < LIGNES and 0 <= c < COLONNES for l, c in cases):
                    fenetres.append(cases)
    return fenetres


FENETRES = construire_fenetres()

def est_gagnant(grille, joueur):
    """Vrai si joueur a quatre pions alignés quelque part."""
    for fenetre in FENETRES:
        if all(grille[l, c] == joueur for l, c in fenetre):
            return True
    return False

def coup_gagnant(grille, joueur):
    """Une colonne qui fait gagner joueur tout de suite, sinon None."""
    for col in coups_valides(grille):
        if est_gagnant(simuler_coup(grille, col, joueur), joueur):
            return col
    return None

def coups_surs(grille, coups):
    """Les coups après lesquels l'adversaire ne peut pas gagner tout de suite."""
    return [col for col in coups
            if coup_gagnant(simuler_coup(grille, col, MOI), ADV) is None]

class AgentRegles(AgentAleatoire):
    """Gagner, sinon bloquer, sinon un coup sûr de préférence au centre."""

    POIDS = [1, 2, 3, 4, 3, 2, 1]

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        grille = vers_grille(observation)
        coups = colonnes_libres(action_mask)
        for joueur in (MOI, ADV):  # 1. gagner, 2. bloquer
            col = coup_gagnant(grille, joueur)
            if col is not None:
                return col
        candidats = coups_surs(grille, coups) or coups  # 3. ne pas offrir
        poids = [self.POIDS[col] for col in candidats]  # 4. le centre
        return random.choices(candidats, weights=poids)[0]

### Lire du code 1

`pytest` exécute les fonctions dont le nom commence par `test_` dans
les fichiers `test_*.py`. Combien de tests va-t-il exécuter dans ce
fichier, et combien échoueront ?

In [ ]:
%%writefile test_demo.py
def test_cases():
    assert 6 * 7 == 42


def verifie_directions():
    assert 2 + 2 == 5


def test_centre():
    assert 7 // 2 == 3.5

In [ ]:
import subprocess
sortie = subprocess.run([sys.executable, "-m", "pytest", "-q", "test_demo.py"],
                        capture_output=True, text=True)
print(sortie.stdout[-600:])

**Réponse.** Deux tests exécutés (`test_cases`, `test_centre`) ;
`verifie_directions` est ignorée, malgré son assertion fausse, car son
nom ne commence pas par `test_`. `test_centre` échoue : `7 // 2` vaut
`3` (division entière), pas `3.5`.

### Exercice 3 — `test_puissance4.py`

Le fichier de tests importe le module. Deux tests sont fournis ;
ajoutez-en **au moins quatre** (reprenez ceux de l'exercice 1, les
scénarios, et un test « l'agent joue toujours un coup autorisé » sur
des positions tirées au hasard). La cellule suivante lance `pytest`.

In [ ]:
%%writefile test_puissance4.py
import numpy as np

from puissance4 import (ADV, MOI, AgentAleatoire, AgentRegles, coup_gagnant,
                        coups_valides, est_gagnant, grille_depuis_texte,
                        ligne_libre, masque_depuis_grille,
                        observation_depuis_grille, simuler_coup)


def test_grille_vide():
    vide = np.zeros((6, 7), dtype=int)
    assert coups_valides(vide) == list(range(7))
    assert ligne_libre(vide, 3) == 5


def test_simuler_ne_modifie_pas():
    vide = np.zeros((6, 7), dtype=int)
    simuler_coup(vide, 3, MOI)
    assert vide.sum() == 0


def test_colonne_pleine():
    g = np.zeros((6, 7), dtype=int)
    g[:, 2] = [MOI, ADV] * 3
    assert 2 not in coups_valides(g)
    assert ligne_libre(g, 2) is None


def test_victoires_quatre_directions():
    for cases in ([(5, 0), (5, 1), (5, 2), (5, 3)],
                  [(5, 6), (4, 6), (3, 6), (2, 6)],
                  [(5, 0), (4, 1), (3, 2), (2, 3)],
                  [(2, 0), (3, 1), (4, 2), (5, 3)]):
        g = np.zeros((6, 7), dtype=int)
        for l, c in cases:
            g[l, c] = ADV
        assert est_gagnant(g, ADV) and not est_gagnant(g, MOI)


def test_coup_gagnant():
    g = grille_depuis_texte("""
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. O O . . . .
. X X X O . .
""")
    assert coup_gagnant(g, MOI) == 0
    assert coup_gagnant(g, ADV) is None


def test_agent_regles_bloque():
    g = grille_depuis_texte("""
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . X .
. . . . X X .
O O O . X O .
""")
    obs, masque = observation_depuis_grille(g), masque_depuis_grille(g)
    assert all(AgentRegles().choose_action(obs, action_mask=masque) == 3
               for _ in range(10))


def test_agents_jouent_des_coups_autorises():
    rng = np.random.default_rng(0)
    for _ in range(100):
        g = np.zeros((6, 7), dtype=int)
        for k in range(rng.integers(0, 30)):  # une position de jeu au hasard
            if not coups_valides(g) or est_gagnant(g, MOI) or est_gagnant(g, ADV):
                break
            g = simuler_coup(g, rng.choice(coups_valides(g)), [MOI, ADV][k % 2])
        if not coups_valides(g):
            continue
        obs, masque = observation_depuis_grille(g), masque_depuis_grille(g)
        for agent in (AgentAleatoire(), AgentRegles()):
            assert masque[agent.choose_action(obs, action_mask=masque)] == 1


def test_fin_de_partie_renvoie_none():
    obs = np.zeros((6, 7, 2), dtype=np.int8)
    assert AgentRegles().choose_action(obs, terminated=True,
                                       action_mask=[1] * 7) is None

In [ ]:
import re
sortie = subprocess.run([sys.executable, "-m", "pytest", "-q", "test_puissance4.py"],
                        capture_output=True, text=True)
print(sortie.stdout[-1500:])
reussis = re.search(r"(\d+) passed", sortie.stdout)
assert sortie.returncode == 0, "des tests échouent : lisez le rapport ci-dessus"
assert reussis and int(reussis.group(1)) >= 6, "au moins six tests"
print("✅ Exercice 3 réussi")

L'avant-dernier test fabrique 100 positions en jouant des coups au
hasard, et vérifie que chaque agent respecte le masque sur chacune.
La graine fixe (`default_rng(0)`) rend le test reproductible : un test
qui échoue une fois sur dix ne sert à rien.

`pylint` note la qualité du code sur 10 : noms, docstrings, longueur
des lignes, imports inutilisés… Lisez ses remarques et corrigez ce qui
a du sens dans votre propre dépôt.

In [ ]:
try:
    import pylint
except ImportError:  # Colab : pylint n'est pas préinstallé
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pylint"],
                   check=True)
sortie = subprocess.run([sys.executable, "-m", "pylint", "puissance4.py",
                         "--disable=too-few-public-methods"],
                        capture_output=True, text=True)
print(sortie.stdout[-1500:])

---

## 3. Le temps et la mémoire

Sur ML-Arena, chaque appel à `choose_action` doit répondre en moins de
**0.25 s**, aller-retour compris ; au-delà, le match est perdu.
Il faut donc mesurer, et garder une marge.

### Exercice 4 — `AgentChronometre`

Écrivez une classe qui **enveloppe** un agent : elle respecte le même
contrat, transmet chaque appel à l'agent enveloppé, et enregistre la
durée de chaque `choose_action` dans `self.durees`
(`time.perf_counter()` avant et après). C'est la *délégation* : on
ajoute un comportement sans toucher à l'agent.

In [ ]:
class AgentChronometre:
    """Enveloppe un agent et mesure la durée de chaque choose_action."""

    def __init__(self, agent):
        self.agent = agent
        self.durees = []

    def setup(self, observation_space, action_space):
        return self.agent.setup(observation_space, action_space)

    def reset(self, env_player_name, episode_index):
        return self.agent.reset(env_player_name, episode_index)

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        debut = time.perf_counter()
        action = self.agent.choose_action(observation, reward, terminated,
                                          truncated, info, action_mask)
        self.durees.append(time.perf_counter() - debut)
        return action

In [ ]:
class _Lent(AgentAleatoire):
    def choose_action(self, *args, **kwargs):
        time.sleep(0.01)
        return super().choose_action(*args, **kwargs)

chrono = AgentChronometre(_Lent())
bilan = jouer_match(chrono, AgentAleatoire(), 4)
assert len(chrono.durees) > 4 * 3, "une durée par coup joué"
assert min(chrono.durees) >= 0.01
assert chrono.agent.env_player_name in ("player_0", "player_1"), "reset transmis"
print("✅ Exercice 4 réussi")

### Lire du code 2

`AgentChronometre` **contient** un agent, il n'en **hérite** pas. Que
vont afficher ces trois tests ?

In [ ]:
chrono = AgentChronometre(AgentRegles())
print(isinstance(chrono, AgentAleatoire))
print(hasattr(chrono, "POIDS"))
print(hasattr(chrono.agent, "POIDS"))

**Réponse.** `False`, `False`, `True`. L'enveloppe n'est pas un
`AgentAleatoire` et n'a pas les attributs de l'agent ; elle les
atteint par `self.agent`. La délégation permet d'envelopper **n'importe
quel** agent — aléatoire, à règles, minimax — avec la même classe, là
où l'héritage demanderait une sous-classe chronométrée par agent.

In [ ]:
for nom, agent in [("aléatoire", AgentAleatoire()), ("pondéré", AgentPondere()),
                   ("règles", AgentRegles())]:
    chrono = AgentChronometre(agent)
    jouer_match(chrono, AgentAleatoire(), 20)
    d = np.array(chrono.durees) * 1000
    print(f"{nom:10s} moyenne {d.mean():6.2f} ms   max {d.max():6.2f} ms")

### Exercice 5 — La mémoire

`tracemalloc` (bibliothèque standard) mesure la mémoire allouée par
Python. Écrivez `pic_memoire(agent)` : démarre la mesure
(`tracemalloc.start()`), joue une partie contre `AgentAleatoire`,
lit le pic (`tracemalloc.get_traced_memory()[1]`, en octets), arrête
la mesure (`tracemalloc.stop()`) et renvoie le pic en **Mo**.

In [ ]:
import tracemalloc


def pic_memoire(agent):
    """Le pic de mémoire (Mo) pendant une partie contre AgentAleatoire."""
    tracemalloc.start()
    jouer_partie(agent, AgentAleatoire())
    pic = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return pic / 1e6

In [ ]:
mo = pic_memoire(AgentRegles())
assert isinstance(mo, float) and 0 < mo < 100
print(f"pic pendant une partie : {mo:.2f} Mo")
print("✅ Exercice 5 réussi")

---

## 4. Le tournoi

### Exercice 6 — `tournoi`

Chaque paire d'agents joue un match (`jouer_match`, places
alternées). Renvoyez un `DataFrame` dont la case (ligne `a`, colonne
`b`) est le taux de victoire de `a` contre `b` ; la diagonale reste
`NaN`. `itertools.combinations(noms, 2)` donne les paires.

In [ ]:
import itertools

import pandas as pd


def tournoi(agents, n_parties=50):
    """Chaque paire d'agents joue un match ; taux de victoire ligne / colonne."""
    noms = list(agents)
    taux = pd.DataFrame(np.nan, index=noms, columns=noms)
    for a, b in itertools.combinations(noms, 2):
        bilan = jouer_match(agents[a], agents[b], n_parties)
        taux.loc[a, b] = bilan["victoires"] / n_parties
        taux.loc[b, a] = bilan["defaites"] / n_parties
    return taux

In [ ]:
agents = {"aléatoire": AgentAleatoire(), "pondéré": AgentPondere(),
          "règles": AgentRegles()}
taux = tournoi(agents, 50)
print(taux.round(2))
assert list(taux.index) == list(agents) and taux.isna().sum().sum() == 3
for a, b in itertools.combinations(agents, 2):
    assert taux.loc[a, b] + taux.loc[b, a] <= 1, "victoires + défaites ≤ 1"
assert taux.loc["règles", "aléatoire"] >= 0.9
print("✅ Exercice 6 réussi")

### Trouvez l'erreur

Ce match oppose deux agents **identiques**. Pourquoi l'agent A gagne-t-il
nettement plus d'une partie sur deux ? Corrigez `match_biaise` dans
`match_equitable`.

In [ ]:
def match_biaise(agent_a, agent_b, n_parties):
    victoires = 0
    for i in range(n_parties):
        resultat, _ = jouer_partie(agent_a, agent_b, index=i)
        victoires += resultat > 0
    return victoires / n_parties

print(match_biaise(AgentAleatoire(), AgentAleatoire(), 300))

**Réponse.** `a_commence` vaut `True` par défaut : A commence
**toutes** les parties, et celui qui commence a un avantage (environ
55 % contre un aléatoire). Le taux mesure la place, pas l'agent. Il
faut alterner, comme le fait `jouer_match` (et comme ML-Arena change
les places d'une partie à l'autre).

### Exercice 7 — `match_equitable`

Écrivez la version corrigée : `agent_a` commence une partie sur deux.

In [ ]:
def match_equitable(agent_a, agent_b, n_parties):
    victoires = 0
    for i in range(n_parties):
        resultat, _ = jouer_partie(agent_a, agent_b, a_commence=(i % 2 == 0),
                                   index=i)
        victoires += resultat > 0
    return victoires / n_parties

In [ ]:
class _Compteur(AgentAleatoire):
    def reset(self, env_player_name, episode_index):
        self.premieres = getattr(self, "premieres", 0) + (env_player_name == "player_0")
        return super().reset(env_player_name, episode_index)

compteur = _Compteur()
match_equitable(compteur, AgentAleatoire(), 20)
assert compteur.premieres == 10, "A doit commencer une partie sur deux"
print("✅ Exercice 7 réussi")

### Exercice 8 — Quelle règle sert le plus ?

Pour comprendre un agent, on journalise ses décisions. Écrivez
`AgentReglesJournal(AgentRegles)` : même logique, mais chaque décision
incrémente `self.journal[raison]` (un `collections.Counter`), avec
`raison` parmi `"gagner"`, `"bloquer"`, `"centre"`. Dans un vrai
projet, on écrirait ces lignes dans un fichier avec le module
`logging` (ou la bibliothèque `loguru`) plutôt que de les compter.

In [ ]:
from collections import Counter


class AgentReglesJournal(AgentRegles):
    """AgentRegles qui compte la raison de chaque décision."""

    def __init__(self):
        self.journal = Counter()

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        grille = vers_grille(observation)
        coups = colonnes_libres(action_mask)
        for joueur, raison in ((MOI, "gagner"), (ADV, "bloquer")):
            col = coup_gagnant(grille, joueur)
            if col is not None:
                self.journal[raison] += 1
                return col
        self.journal["centre"] += 1
        candidats = coups_surs(grille, coups) or coups
        poids = [self.POIDS[col] for col in candidats]
        return random.choices(candidats, weights=poids)[0]

In [ ]:
journal = AgentReglesJournal()
bilan = jouer_match(journal, AgentRegles(), 40)
print(dict(journal.journal), "|", bilan["victoires"], "victoires,",
      bilan["defaites"], "défaites")
assert set(journal.journal) <= {"gagner", "bloquer", "centre"}
assert journal.journal["gagner"] >= bilan["victoires"] > 0
print("✅ Exercice 8 réussi")

---

## 5. Pour aller plus loin (facultatif)

### Exercice 9 — Combien de parties pour conclure ?

Un taux de victoire mesuré sur `n` parties fluctue. L'intervalle de
confiance à 95 % (approximation normale) est :

$$\hat p \pm 1{,}96\sqrt{\hat p(1-\hat p)/n}$$

Écrivez `intervalle_confiance(victoires, n)` qui renvoie
`(bas, haut)`, bornés à [0, 1]. Combien de parties faut-il pour
distinguer 55 % de 50 % ?

In [ ]:
def intervalle_confiance(victoires, n):
    """Intervalle de confiance à 95 % d'un taux de victoire."""
    p = victoires / n
    marge = 1.96 * np.sqrt(p * (1 - p) / n)
    return max(0.0, p - marge), min(1.0, p + marge)


for n in (10, 100, 400, 1000):
    bas, haut = intervalle_confiance(0.55 * n, n)
    print(f"55 % sur {n:4d} parties : [{bas:.1%}, {haut:.1%}]")

In [ ]:
bas, haut = intervalle_confiance(50, 100)
assert abs(bas - 0.402) < 0.001 and abs(haut - 0.598) < 0.001
assert intervalle_confiance(100, 100) == (1.0, 1.0)
print("✅ Exercice 9 réussi")

**Lecture.** Sur 100 parties, 55 % donne [45 %, 65 %] : impossible de
dire que l'agent fait mieux que 50 %. Il faut environ 400 parties pour
que l'intervalle exclue 50 %. Un match de 10 parties sur ML-Arena est
très bruité : c'est pourquoi le classement ELO agrège de nombreux
matchs.

---

**Bilan.** Vos agents sont testés, chronométrés et comparés. Le lab 4
construit un agent qui anticipe plusieurs coups (minimax), sous la
contrainte de temps de la plateforme, et le soumet sur ML-Arena.